# Pipeline
This notebook runs the end-to-end pipeline, Raw → Bronze → Silver → Gold, using the functions in `scripts/`. The findings and assumptions behind each rule are in `00_data_exploration`.

## Architecture
```
Order.json    ─► bronze.orders    ─► silver.orders    ─┐
Customer.xlsx ─► bronze.customers ─► silver.customers ─┼─► gold.enriched_orders ─► gold.profit_agg ─► SQL aggregates
Product.csv   ─► bronze.products  ─► silver.products  ─┘
                                     silver.product_classification (lookup)
Landing[./Data/]     BRONZE                 SILVER                          GOLD
```

## Contents
1. Setup: configurations and imports
2. Task 1: Create raw tables for each source dataset (Landing → Bronze)
3. Task 2: Create an enriched table for customers and products  (Bronze → Silver)
4. Task 3: 	Create an enriched table which has
-       a. order information : Profit rounded to 2 decimal places
-       b. Customer name and country
-       c. Product category and sub category

5. Task 4: Create an aggregate table that shows profit by 
-                                  1.	Year
-                                  2.	Product Category
-                                  3.	Product Sub Category
-                                  4.	Customer

6. Task 5: SQL aggregates

## Assumptions
1. Raw Input files are expected to be in the data folder of the project repo

2. Bronze Schema stores the source data as-is, with spark friendly column names.
3. Data is ingested using overwrite schema pattern for simplicity
3. Customer names and phone numbers of "CUSTOMER TABLE" is cleaned. Invalid phones are set to null.
4. "PRODUCT TABLE" is not de-duplicated as the deduplication strategy needs to be aligned. 
5. "Category" and "sub-category" lookup table based on Product_id is created and used for aggregations
6."ORDERS TABLE" is left-joined to "CUSTOMERS" so no orders are dropped.
7. `order_year` is derived from `order_date`.

# Key Configurations

In [0]:
%pip install openpyxl

In [0]:
%restart_python

In [0]:
import os
from pyspark.sql import functions as F
from pyspark.sql.types import  StructType, StringType, StructField, IntegerType
import re, pandas as pd
from scripts.cleaning import clean_cust_name , clean_phone
from scripts.utils import *
from scripts.dq import *
from scripts.transforms import round_profit


#Catalog & Schema
CATALOG= "DE_ASSESSMENT"
SCHEMA_BRONZE = "BRONZE" 
SCHEMA_SILVER = "SILVER" 
SCHEMA_GOLD = "GOLD" 

#Input file paths 
file_name_customer ="Customer.xlsx"
file_name_orders ="Orders.json"
file_name_product ="Products.csv"

#row counts manually extracted for validation
row_count_customer = 793
row_count_orders = 9994
row_count_products = 1851

#Raw data volume path
input_folder_path = os.path.join(os.getcwd(), "data")

#Column names of the input tables. Calculated using cell 4 output of 00_exploratory.ipynb . Used for schema validation
columns_names_orders = ['Customer ID','Discount', 'Order Date', 'Order ID', 'Price', 'Product ID', 'Profit', 'Quantity', 'Row ID', 'Ship Date', 'Ship Mode']
columns_names_products = ['Product ID', 'Category', 'Sub-Category', 'Product Name', 'State', 'Price per product']
columns_names_customer = ['Customer ID', 'Customer Name', 'email', 'phone', 'address', 'Segment', 'Country', 'City', 'State', 'Postal Code', 'Region']


In [0]:
# Create the catalog and schema
spark.sql (f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
spark.sql (f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA_BRONZE}")
spark.sql (f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA_SILVER}")
spark.sql (f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA_GOLD}") 

#Task 1: Create raw tables for each source dataset (Landing → Bronze)

In [0]:

raw_orders = (spark.read.option("multiLine", True)
                        .option("mode","PERMISSIVE")
                        .option("rescuedDataColumn","_rescued_data")
                        .option("primitivesAsString",True)
                        .json(f"{input_folder_path}/{file_name_orders}")
                )

raw_products = (spark.read.option("header",True)
                        .option("enforceSchema",False)
                        .option("rescuedDataColumn","_rescued_data")
                        .option("mode","PERMISSIVE")
                        .option("primitivesAsString",True)                        
                        .csv(f"{input_folder_path}/{file_name_product}")
                )


raw_pdf_customer =pd.read_excel(f"{input_folder_path}/{file_name_customer}",'Worksheet',dtype ="str")

raw_pdf_customer = raw_pdf_customer.astype("object").where(raw_pdf_customer.notna(),None)
raw_customer = spark.createDataFrame(raw_pdf_customer)



In [0]:
dq_check_schema(raw_orders.columns,columns_names_orders)
dq_check_schema(raw_customer.columns,columns_names_customer)
dq_check_schema(raw_products.columns,columns_names_products)

dq_check_rows(raw_orders.count(),row_count_orders)
dq_check_rows(raw_customer.count(),row_count_customer)
dq_check_rows(raw_products.count(),row_count_products)

bronze_orders=raw_orders.toDF(*rename(raw_orders.columns))
bronze_customer = raw_customer.toDF(*rename(raw_customer.columns))
bronze_products = raw_products.toDF(*rename(raw_products.columns))


bronze_jobs = [
    (bronze_orders,   "orders",   file_name_orders),
    (bronze_customer, "customer", file_name_customer),
    (bronze_products, "products", file_name_product),
]

for df, table, file_name in bronze_jobs:
    print(to_bronze(df, f"{CATALOG}.{SCHEMA_BRONZE}.{table}", f"{input_folder_path}/{file_name}"))


# Task 2: Create an enriched table for customers and products  (Bronze → Silver)


In [0]:
bronze_customer = spark.sql(f"select * from {CATALOG}.{SCHEMA_BRONZE}.customer")

# Applying data cleaning logics on Customer name and Phone number
silver_customer = (bronze_customer.withColumn("customer_name",clean_cust_name("customer_name"))
                                  .withColumn("phone",clean_phone("phone"))
               
               )

silver_customer.write.mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA_SILVER}.customer")

In [0]:

bronze_product = spark.sql(f"select * from {CATALOG}.{SCHEMA_BRONZE}.products")

# applying type conversions on product table
silver_product= (bronze_product.withColumn("price_per_product",(F.col("price_per_product").try_cast("Decimal(10,2)")))
            )

parts = F.split(F.col("product_id"), "-")

# creating a product classification lookup table
silver_product_classification = (bronze_product
                                        .select( "category", "sub_category","product_id")
                                        .withColumn("product_category_short",     parts.getItem(0))   
                                        .withColumn("product_sub_category_short", parts.getItem(1))
                                        .select(F.col("category").alias("product_category"), F.col("sub_category").alias("product_sub_category"),"product_category_short","product_sub_category_short")
                                        .distinct()
                                )

#writing product table to Silver Schema
(
    silver_product.write
                .mode("overwrite")
                .option("overwriteSchema", True)
                .saveAsTable(f"{CATALOG}.{SCHEMA_SILVER}.products")
)

#writing product classification lookup to Silver Schema
(
    silver_product_classification.write
                                 .mode("overwrite")
                                 .option("overwriteSchema", True)
                                 .saveAsTable(f"{CATALOG}.{SCHEMA_SILVER}.product_classification_lookup")
    )

# Task 3: Create an enriched table which has
                          1. order information : Profit rounded to 2 decimal places
                          2. Customer name and country
                          3. Product category and sub category

In [0]:
bronze_orders = spark.sql(f"select * from {CATALOG}.{SCHEMA_BRONZE}.orders")
parts = F.split(F.col("product_id"), "-")
silver_orders = (bronze_orders.withColumn("discount",F.col("discount").try_cast("double"))
                     .withColumn("price",F.col("price").try_cast("Decimal(10,2)"))
                     .withColumn("quantity",F.col("quantity").try_cast("int"))
                    .withColumn("profit",F.col("profit").try_cast("double"))
                    .withColumn("order_date",F.to_date(F.col("order_date"),"d/M/yyyy"))
                    .withColumn("order_year",F.year(F.col("order_date")))
                    .withColumn("ship_date",F.to_date(F.col("ship_date"),"d/M/yyyy"))
                    .withColumn("price_per_product",F.col("price")/F.col("quantity").try_cast("Decimal(10,2)"))
                    .withColumn("product_category_short",     parts.getItem(0))   
                    .withColumn("product_sub_category_short", parts.getItem(1))


            )
silver_orders.write.mode("overwrite").option("overwriteSchema", True).saveAsTable(f"{CATALOG}.{SCHEMA_SILVER}.orders")

In [0]:

silver_orders = spark.sql(f"""select customer_id
                                    ,discount
                                    ,order_date
                                    ,order_year
                                    ,order_id,price
                                    ,product_id,profit
                                    ,quantity,ship_date
                                    ,ship_mode
                                    ,product_category_short
                                    ,product_sub_category_short 
                                from {CATALOG}.{SCHEMA_SILVER}.orders""")
# Rounding the profit umbers to 2 decimal
silver_orders = round_profit(silver_orders)
parts = F.split(F.col("product_id"), "-")

# reading the product classification table
silver_product_classification_lookup  =  spark.sql(f"""select product_category
                                                        , product_sub_category
                                                        ,product_category_short
                                                        ,product_sub_category_short 
                                                from {CATALOG}.{SCHEMA_SILVER}.product_classification_lookup""")
# reading the customer table
silver_customers = spark.sql(f"""select customer_id
                                     ,customer_name
                                     ,country 
                                from {CATALOG}.{SCHEMA_SILVER}.customer""")

# Joing the order table with the product classification table and customer table
orders_enriched = (silver_orders.join(silver_customers, 
                                      on= "customer_id", 
                                      how="left")
                                .join(silver_product_classification_lookup, 
                                      on = ["product_category_short","product_sub_category_short"],
                                      how= "left"))
# writing the enriched order table to the gold database
orders_enriched.write.mode("overwrite").option("overwriteSchema", True).saveAsTable(f"{CATALOG}.{SCHEMA_GOLD}.orders_enriched")

display(orders_enriched)

#Task 4: Create an aggregate table that shows profit by
                             1.	Year
                             2.	Product Category
                             3.	Product Sub Category
                             4.	Customer

In [0]:
gold_profit_summary=( orders_enriched.groupBy("order_year"
                                            ,"product_category"
                                            ,"product_sub_category"
                                            ,"customer_id"
                                            ,"customer_name")
                                    .agg(F.sum("profit").alias("total_profit"))
                                    .orderBy("order_year"
                                             ,"product_category"
                                             ,"product_sub_category"
                                             ,"customer_id"
                                             ,"customer_name"))

(
    gold_profit_summary.write
                        .mode("overwrite")
                        .option("overwriteSchema", True)
                        .saveAsTable(f"{CATALOG}.{SCHEMA_GOLD}.profit_summary")
                        )




In [0]:
#Testing the aggregations
s = spark.sql(f"""
  select (select sum(profit)       from {CATALOG}.silver.orders)        as silver,
         (select sum(profit)       from {CATALOG}.gold.orders_enriched) as enriched,
         (select sum(total_profit) from {CATALOG}.gold.profit_summary)  as summary
""").first()
print(s)
assert abs(s.summary - s.enriched) < 0.01, "aggregate does not match enriched"
assert abs(s.enriched - s.silver) < 1.00,  "enriched profit differs from silver beyond rounding"

# Task 5.	Using SQL output the following aggregates
1.	Profit by Year
2.	Profit by Year + Product Category
3.	Profit by Customer
4.	Profit by Customer + Year


In [0]:
%sql
-- Task 5.1 Profit by Year
select  order_year
        ,cast(sum(total_profit) as decimal(10,2)) as total_profit 
from de_assessment.gold.profit_summary
group by order_year
order by order_year

In [0]:
%sql
-- Task 5.2 Profit by Year + Product Category
select  order_year
        ,product_category
        ,cast(sum(total_profit) as decimal(10,2)) as total_profit 
from de_assessment.gold.profit_summary
group by order_year, product_category
order by order_year, total_profit 

In [0]:
%sql
-- Task 5.3 Profit by Customer
select  customer_id
        ,customer_name
        ,cast(sum(total_profit) as decimal(10,2)) as total_profit 
from de_assessment.gold.profit_summary
group by customer_id
        ,customer_name
order by total_profit desc

In [0]:
%sql
-- Task 5.4 Profit by Customer + Year
select   order_year
        ,customer_id
        ,customer_name
        ,cast(sum(total_profit) as decimal(10,2)) as total_profit 
from de_assessment.gold.profit_summary
group by order_year
        ,customer_id
        ,customer_name
order by order_year, total_profit desc

### End of _Workbook_